# PPI GovTech · [NOME DA SOLUÇÃO]
**Grupo:** [nomes] · **Aprendizagem Computacional · 4º TIA**

Este é o esqueleto do notebook do MVP. Ele segue a mesma ordem do **PPI-molde (Comparece MT)**. Abram os dois lado a lado: cada seção aqui tem a seção correspondente lá, já resolvida.

**Regras do jogo**
- Cada `TODO` é uma decisão do grupo. Não apaguem as perguntas: respondam abaixo delas.
- O notebook precisa rodar de cima para baixo, sem erro, em um Colab novo (*Executar tudo*).
- `random_state=42` em tudo que sorteia.

| Seção | Entrega | Data |
|---|---|---|
| 1 | Pergunta, decisor, X e y | 03/11 |
| 2–3 | Base analítica v0 + EDA | 10/11 |
| 4 | Baseline × modelo | 17/11 |
| 5–7 | Comparação, previsões e decisão v0 | 24/11 |
| 8–9 | MVP + checklist | 01/12 |

## 1 · Entendimento do negócio (03/11)

| Item | Resposta do grupo (uma frase) |
|---|---|
| Problema público | TODO |
| Decisor (quem usa) | TODO |
| Decisão apoiada (que ação muda?) | TODO |
| Pergunta analítica | TODO |
| Unidade de análise (o que é uma linha?) | TODO |
| y (alvo) | TODO, ou "não há y: tarefa não supervisionada" |
| X (entradas candidatas) | TODO |
| Tarefa | regressão / classificação / agrupamento + otimização |

**Hipóteses:** H1: TODO · H2: TODO · H3: o modelo erra menos que o baseline.

## 2 · Dados: fonte, ETL e validação (10/11)
**Fontes** (link, órgão e data do download): TODO

Coloquem aqui o ETL: leitura dos arquivos brutos → validação → transformação → `base_analitica.csv`. Se a base for grande, o ETL pode ficar num script separado no repositório, e este notebook lê o CSV final por URL (*raw* do GitHub).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMENTE = 42
# Arquivos gerados ficam FORA da pasta do projeto: /content no Colab, sua pasta de usuário no laboratório
SAIDA = (Path('/content') if Path('/content').exists() else Path.home() / 'ac_saida') / 'comparece_mt'
SAIDA.mkdir(parents=True, exist_ok=True)
print('Arquivos gerados vão para:', SAIDA)

# TODO: troque pelo CSV do seu grupo (na pasta dados/ do repositório) e pelo link raw dele no GitHub
ARQUIVO = 'base_analitica.csv'
URL = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/ppi_molde_comparece_mt/dados/' + ARQUIVO

# Procura a base na pasta dados/ do projeto; no Colab (aberto pelo GitHub), baixa do repositório
candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
local = next((p for p in candidatos if p.exists()), None)
base = pd.read_csv(local if local is not None else URL, dtype={'CD_MUNICIPIO': str})
print('Base lida de:', local.resolve() if local is not None else URL)
print(base.shape)
base.head()

In [ ]:
# Checagens de qualidade: escrevam as regras do seu domínio.
ID = ['TODO_coluna_id']          # nunca entra em X
X_COLS = ['TODO', 'TODO']
Y_COL = 'TODO'

assert base[ID[0]].is_unique, 'unidade de análise repetida'
assert Y_COL not in X_COLS
# TODO: limites (ex.: percentuais entre 0 e 100), contagens >= 0, nulos aceitáveis...
print(base[X_COLS + [Y_COL]].isna().sum())

### Dicionário mínimo
| Variável | Descrição | Tipo | Fonte | Papel (X / y / ID / descarte) | Tratamento |
|---|---|---|---|---|---|
| TODO | | | | | |

**Pergunta de vazamento:** alguma coluna só é conhecida **depois** do que queremos prever, ou é parte do cálculo de y? Se sim, ela sai de X. Resposta do grupo: TODO

In [ ]:
# EDA mínima: distribuição de y + relação de cada X com y
fig, eixos = plt.subplots(1, len(X_COLS) + 1, figsize=(4 * (len(X_COLS) + 1), 3.5))
eixos[0].hist(base[Y_COL], bins=20); eixos[0].set_title('Distribuição de y')
for ax, col in zip(eixos[1:], X_COLS):
    ax.scatter(base[col], base[Y_COL], s=12, alpha=0.6); ax.set_title(col)
plt.tight_layout(); plt.show()

**O que a EDA mostrou?** (3 frases, sem afirmar causa): TODO

## 3 · Separar antes de aprender

In [ ]:
from sklearn.model_selection import train_test_split
X, y = base[X_COLS], base[Y_COL]
# Classificação: acrescente stratify=y
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.20, random_state=SEMENTE)
print(len(X_treino), len(X_teste))

## 4 · Baseline × primeiro modelo (17/11)
**Baseline mais exigente:** além do `Dummy`, existe uma regra que o gestor usaria sem modelo (repetir o valor do ano anterior, por exemplo)? Se existir, ela também entra na tabela, como a persistência do PPI-molde.

**Métrica escolhida e por quê** (ligue ao custo do erro na decisão): TODO

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression

cv = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)   # classificação: StratifiedKFold
METRICA = 'neg_mean_absolute_error'                          # classificação: 'f1', 'recall', 'average_precision'...

def avaliar_cv(modelo, nome):
    r = cross_validate(modelo, X_treino, y_treino, cv=cv, scoring=METRICA)
    return {'modelo': nome, 'media_cv': r['test_score'].mean(), 'desvio_cv': r['test_score'].std()}

candidatos = {
    'Baseline': DummyRegressor(strategy='mean'),   # TODO: DummyClassifier(strategy='most_frequent')
    'Modelo 1': Pipeline([('escala', StandardScaler()), ('modelo', LinearRegression())]),  # TODO
}
pd.DataFrame([avaliar_cv(m, n) for n, m in candidatos.items()])

## 5 · Comparação justa (24/11)
Acrescentem 1 ou 2 modelos ao dicionário `candidatos` (árvore, Random Forest, k-NN...), mantendo o mesmo `cv` e a mesma métrica. Escolham **antes** de abrir o teste.

**Modelo escolhido e por quê:** TODO

In [ ]:
from sklearn.metrics import mean_absolute_error  # classificação: classification_report, confusion_matrix

escolhido = 'Modelo 1'  # TODO
modelo_final = candidatos[escolhido].fit(X_treino, y_treino)
baseline_final = candidatos['Baseline'].fit(X_treino, y_treino)
# O teste é usado UMA vez, aqui.
print('Baseline:', mean_absolute_error(y_teste, baseline_final.predict(X_teste)))
print('Modelo  :', mean_absolute_error(y_teste, modelo_final.predict(X_teste)))

## 6 · Previsões fora da amostra para todas as linhas

In [ ]:
from sklearn.model_selection import cross_val_predict
base['PREVISAO'] = cross_val_predict(candidatos[escolhido], X, y, cv=cv)
# Classificação com probabilidade: cross_val_predict(..., method='predict_proba')[:, 1]

## 7 · Da previsão à decisão (24/11)
| Item | Resposta do grupo |
|---|---|
| Recurso limitado | TODO (orçamento, equipes, vagas, km...) |
| Benefício de atender uma linha | TODO (fórmula que usa a PREVISAO) |
| Custo de atender uma linha | TODO |
| Restrição | TODO |
| Método | mochila / caminho mínimo / fluxo / PLI / ranking simples |

**Hipóteses declaradas** (se os parâmetros não forem oficiais): TODO

In [ ]:
# TODO: calcular BENEFICIO e CUSTO e escolher a carteira.
# A função mochila() do PPI-molde pode ser copiada como está.
# Comparem a solução com pelo menos uma regra simples (ex.: "maior previsão primeiro").

## 8 · MVP: exportar para a vitrine (01/12)

In [ ]:
base.to_csv(SAIDA / 'previsoes.csv', index=False, encoding='utf-8-sig')   # fonte do Power BI ou do dashboard web
# TODO: salvar a carteira em SAIDA / 'ranking_decisao.csv'
# TODO (web): adaptar o bloco do dashboard HTML do PPI-molde

## 9 · Checklist das 7 perguntas da banca
| Pergunta | Resposta do grupo |
|---|---|
| 1. Qual é o y? | TODO |
| 2. Qual é o baseline? | TODO |
| 3. Por que essa métrica? | TODO |
| 4. Onde foi feito o split? | TODO |
| 5. Há vazamento? | TODO |
| 6. A conclusão tem evidência? | TODO (números: baseline × modelo) |
| 7. Roda do zero? | TODO (outro grupo rodou seguindo só o README?) |

**Limitações:** TODO